# 72H Journal Revision — Clean Main Branch + Physics-Informed DLSTM

This notebook runs the cleaned **72h main evolution-aware branch** and adds a **Physics-Informed / Physics-Regularised DLSTM (PI-DLSTM)** extension.

It uses:

1. 16 SHARP magnetic features.
2. Magnetic-evolution features, ΔX, computed within each active region.
3. 7-step chronological sequences.
4. LSTM, BiLSTM, Transformer, DLSTM.
5. Simple/weighted/stacking ensembles.
6. PI-DLSTM using an auxiliary physics-proxy loss based only on past/current SHARP magnetic quantities.
7. Validation-only threshold tuning and validation-selected weighted ensembles.
8. Bootstrap confidence intervals and McNemar comparisons.

Important: this is **physics-informed/physics-regularised**, not a full MHD PDE-PINN. It is safe to describe as a physics-informed temporal extension using SHARP physical proxies.

In [1]:
# ============================================================
# 0. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# 1. Imports and reproducibility
# ============================================================

!pip -q install tensorflow==2.19.0 scikit-learn joblib

import os, random, joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score
)
from sklearn.linear_model import LogisticRegression

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Seed fixed:", SEED)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 807.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 48.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
TensorFlow: 2.19.0
Seed fixed: 42


In [3]:
# ============================================================
# 2. Experiment configuration
# ============================================================

DATA_PATH = "/content/drive/MyDrive/AR_Stratified/HMI_AR_2010_2025_ML_READY_16_LABELED_MX3D.csv"
OUT_DIR = "/content/drive/MyDrive/AR_Stratified/Journal_Revision_72h_PI_DLSTM_CLEAN"
os.makedirs(OUT_DIR, exist_ok=True)

TIME_COL = "T_REC_dt"
AR_COL = "NOAA_AR"
LABEL_COL = "label_MX_3d"

LOOKBACK = 7
MA_WINDOW = 5
EPOCHS = 20
BATCH_SIZE = 128

MAG_FEATURES = [
    "MEANGBZ", "MEANGAM", "MEANGBT", "MEANGBH", "MEANJZD",
    "TOTUSJZ", "MEANALP", "MEANJZH", "ABSNJZH", "SAVNCPP",
    "MEANSHR", "SHRGT45", "R_VALUE", "USFLUX", "TOTPOT", "TOTUSJH"
]

print("DATA_PATH:", DATA_PATH)
print("OUT_DIR:", OUT_DIR)
print("LOOKBACK:", LOOKBACK)
print("MA_WINDOW:", MA_WINDOW)
print("Number of magnetic features:", len(MAG_FEATURES))

DATA_PATH: /content/drive/MyDrive/AR_Stratified/HMI_AR_2010_2025_ML_READY_16_LABELED_MX3D.csv
OUT_DIR: /content/drive/MyDrive/AR_Stratified/Journal_Revision_72h_PI_DLSTM_CLEAN
LOOKBACK: 7
MA_WINDOW: 5
Number of magnetic features: 16


In [4]:
# ============================================================
# 3. Load, clean, and sort dataset
# ============================================================

df = pd.read_csv(DATA_PATH)
print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())

required_cols = [TIME_COL, AR_COL, LABEL_COL] + MAG_FEATURES
missing_cols = [c for c in required_cols if c not in df.columns]
assert len(missing_cols) == 0, f"Missing columns: {missing_cols}"

df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
for col in MAG_FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=required_cols).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)
df = df.sort_values([AR_COL, TIME_COL]).reset_index(drop=True)

print("Clean shape:", df.shape)
display(df[[TIME_COL, AR_COL, LABEL_COL] + MAG_FEATURES[:5]].head())

Raw shape: (22355, 21)
Columns: ['T_REC_dt', 'year', 'NOAA_AR', 'MEANGBZ', 'MEANGAM', 'MEANGBT', 'MEANGBH', 'MEANJZD', 'TOTUSJZ', 'MEANALP', 'MEANJZH', 'ABSNJZH', 'SAVNCPP', 'MEANSHR', 'SHRGT45', 'R_VALUE', 'USFLUX', 'TOTPOT', 'TOTUSJH', 'label_MX_3d', 'future_max_class']
Clean shape: (22130, 21)


,T_REC_dt,NOAA_AR,label_MX_3d,MEANGBZ,MEANGAM,MEANGBT,MEANGBH,MEANJZD
0,2010-05-03 12:00:00,11063,0,163.413,33.823,157.335,82.652,0.853483
1,2010-05-04 12:00:00,11063,0,114.931,23.572,111.969,48.270,-0.446712
2,2010-05-05 12:00:00,11063,0,87.534,28.006,79.667,40.057,-0.065026
3,2010-05-01 12:00:00,11064,0,135.426,34.823,136.321,66.673,0.757142
4,2010-05-02 12:00:00,11064,0,129.025,29.608,127.948,56.867,1.981149


In [5]:
# ============================================================
# 4. Cadence, lookback, and active-region diagnostics
# ============================================================

cadence_hours = (
    df.sort_values([AR_COL, TIME_COL])
      .groupby(AR_COL)[TIME_COL]
      .diff()
      .dt.total_seconds() / 3600.0
).dropna()

print("Cadence diagnostics in hours:")
display(cadence_hours.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

median_cadence = float(cadence_hours.median())
approx_span = (LOOKBACK - 1) * median_cadence

print(f"Median cadence: {median_cadence:.2f} hours")
print(f"Approximate physical span of {LOOKBACK} sampled states = ({LOOKBACK}-1) × {median_cadence:.2f}h = {approx_span:.2f}h")
print(f"Rows with gaps > 2 × median cadence: {int((cadence_hours > 2 * median_cadence).sum())}")

Cadence diagnostics in hours:


,T_REC_dt
count,19862.000000
mean,31.109858
std,911.471378
min,24.000000
25%,24.000000
50%,24.000000
75%,24.000000
90%,24.000000
95%,24.000000
99%,48.000000


Median cadence: 24.00 hours
Approximate physical span of 7 sampled states = (7-1) × 24.00h = 144.00h
Rows with gaps > 2 × median cadence: 38


In [6]:
# ============================================================
# 5. Construct magnetic-evolution features: X + ΔX
# ============================================================

base_X = df[MAG_FEATURES].astype(np.float32)

delta_X = (
    base_X.groupby(df[AR_COL])
          .diff()
          .fillna(0.0)
          .astype(np.float32)
)

DELTA_FEATURES = [f"DELTA_{c}" for c in MAG_FEATURES]
delta_X.columns = DELTA_FEATURES

df_evo = pd.concat(
    [
        df[[TIME_COL, AR_COL, LABEL_COL]].reset_index(drop=True),
        base_X.reset_index(drop=True),
        delta_X.reset_index(drop=True)
    ],
    axis=1
)

EVO_FEATURES = MAG_FEATURES + DELTA_FEATURES
print("Evolution-aware feature count:", len(EVO_FEATURES))
display(df_evo[[TIME_COL, AR_COL, LABEL_COL] + EVO_FEATURES[:8]].head())

Evolution-aware feature count: 32


,T_REC_dt,NOAA_AR,label_MX_3d,MEANGBZ,MEANGAM,MEANGBT,MEANGBH,MEANJZD,TOTUSJZ,MEANALP,MEANJZH
0,2010-05-03 12:00:00,11063,0,163.412994,33.823002,157.335007,82.652000,0.853483,1.032882e+12,0.022914,0.018148
1,2010-05-04 12:00:00,11063,0,114.931000,23.572001,111.969002,48.270000,-0.446712,1.144288e+12,-0.006346,-0.003990
2,2010-05-05 12:00:00,11063,0,87.533997,28.006001,79.667000,40.056999,-0.065026,4.168505e+12,-0.004862,-0.002102
3,2010-05-01 12:00:00,11064,0,135.425995,34.823002,136.320999,66.672997,0.757142,2.411247e+12,-0.011185,-0.003109
4,2010-05-02 12:00:00,11064,0,129.024994,29.608000,127.947998,56.867001,1.981149,1.018732e+12,-0.025063,-0.005277


In [7]:
# ============================================================
# 6. Build 72h lookback sequences
# ============================================================

def build_sequences(frame, feature_cols, lookback):
    X_seq, y_seq, t_seq, ar_seq = [], [], [], []

    for ar, group in frame.groupby(AR_COL):
        group = group.sort_values(TIME_COL).reset_index(drop=True)
        values = group[feature_cols].values.astype(np.float32)
        labels = group[LABEL_COL].values.astype(int)
        times = group[TIME_COL].values

        if len(group) < lookback:
            continue

        for i in range(lookback - 1, len(group)):
            X_seq.append(values[i - lookback + 1:i + 1])
            y_seq.append(labels[i])
            t_seq.append(times[i])
            ar_seq.append(ar)

    return (
        np.array(X_seq, dtype=np.float32),
        np.array(y_seq, dtype=int),
        np.array(t_seq),
        np.array(ar_seq)
    )

X_seq, y_seq, t_seq, ar_seq = build_sequences(df_evo, EVO_FEATURES, LOOKBACK)

print("X_seq:", X_seq.shape)
print("y_seq:", y_seq.shape)
print("Positive count:", int(y_seq.sum()))
print("Positive prevalence:", y_seq.mean())

X_seq: (9438, 7, 32)
y_seq: (9438,)
Positive count: 652
Positive prevalence: 0.06908243271879635


In [9]:
# ============================================================
# 7. Chronological split and scaling
# ============================================================

order = np.argsort(t_seq)
X_seq, y_seq, t_seq, ar_seq = X_seq[order], y_seq[order], t_seq[order], ar_seq[order]

n_total = len(X_seq)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)

X_train_seq = X_seq[:n_train]
y_train_seq = y_seq[:n_train]
X_val_seq = X_seq[n_train:n_train + n_val]
y_val_seq = y_seq[n_train:n_train + n_val]
X_test_seq = X_seq[n_train + n_val:]
y_test_seq = y_seq[n_train + n_val:]

t_train = t_seq[:n_train]
t_val = t_seq[n_train:n_train + n_val]
t_test = t_seq[n_train + n_val:]

ar_train = set(ar_seq[:n_train])
ar_val = set(ar_seq[n_train:n_train + n_val])
ar_test = set(ar_seq[n_train + n_val:])

print("Train:", X_train_seq.shape, "Pos:", int(y_train_seq.sum()))
print("Val  :", X_val_seq.shape, "Pos:", int(y_val_seq.sum()))
print("Test :", X_test_seq.shape, "Pos:", int(y_test_seq.sum()))

print("\nTemporal range:")
print("Train:", pd.to_datetime(t_train.min()), "→", pd.to_datetime(t_train.max()))
print("Val  :", pd.to_datetime(t_val.min()), "→", pd.to_datetime(t_val.max()))
print("Test :", pd.to_datetime(t_test.min()), "→", pd.to_datetime(t_test.max()))

print("\nActive-region overlap across chronological splits:")
print("Train ∩ Val :", len(ar_train & ar_val))
print("Val ∩ Test  :", len(ar_val & ar_test))
print("Train ∩ Test:", len(ar_train & ar_test))

scaler = StandardScaler()
n_steps = X_train_seq.shape[1]
n_features = X_train_seq.shape[2]

X_train_scaled = scaler.fit_transform(X_train_seq.reshape(-1, n_features)).reshape(X_train_seq.shape)
X_val_scaled = scaler.transform(X_val_seq.reshape(-1, n_features)).reshape(X_val_seq.shape)
X_test_scaled = scaler.transform(X_test_seq.reshape(-1, n_features)).reshape(X_test_seq.shape)

joblib.dump(scaler, os.path.join(OUT_DIR, "scaler_72h_main_clean.joblib"))

print("\nScaled shapes:")
print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled  :", X_val_scaled.shape)
print("X_test_scaled :", X_test_scaled.shape)

Train: (6606, 7, 32) Pos: 339
Val  : (1415, 7, 32) Pos: 163
Test : (1417, 7, 32) Pos: 150

Temporal range:
Train: 2010-05-07 12:00:00 → 2023-03-06 12:00:00
Val  : 2023-03-06 12:00:00 → 2024-05-09 12:00:00
Test : 2024-05-09 12:00:00 → 2025-07-16 12:00:00

Active-region overlap across chronological splits:
Train ∩ Val : 4
Val ∩ Test  : 4
Train ∩ Test: 0

Scaled shapes:
X_train_scaled: (6606, 7, 32)
X_val_scaled  : (1415, 7, 32)
X_test_scaled : (1417, 7, 32)


In [10]:
# ============================================================
# 8. Metric helpers
# ============================================================

def tss_hss_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()
    tpr = tp / (tp + fn + 1e-9)
    fpr = fp / (fp + tn + 1e-9)
    tss = tpr - fpr

    numerator = 2 * (tp * tn - fp * fn)
    denominator = ((tp + fn) * (fn + tn) + (tp + fp) * (fp + tn) + 1e-9)
    hss = numerator / denominator
    return float(tss), float(hss)


def tune_threshold_by_tss(y_true, prob):
    thresholds = np.linspace(0, 1, 1001)
    best_thr, best_tss = 0.5, -999.0

    for thr in thresholds:
        y_pred = (prob >= thr).astype(int)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tss, _ = tss_hss_from_cm(cm)
        if tss > best_tss:
            best_tss, best_thr = tss, thr

    return float(best_thr), float(best_tss)


def evaluate_probs_clean(model_name, y_val, val_prob, y_test, test_prob):
    threshold, val_tss = tune_threshold_by_tss(y_val, val_prob)
    y_pred = (test_prob >= threshold).astype(int)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tss, hss = tss_hss_from_cm(cm)
    tn, fp, fn, tp = cm.ravel()

    return {
        "Model": model_name,
        "Threshold": float(threshold),
        "Val_TSS": float(val_tss),
        "Test_TSS": float(tss),
        "HSS": float(hss),
        "ROC_AUC": float(roc_auc_score(y_test, test_prob)),
        "PR_AUC": float(average_precision_score(y_test, test_prob)),
        "Recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "Precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "F1": float(f1_score(y_test, y_pred, zero_division=0)),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)
    }, y_pred


def get_class_weights(y):
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, weights)}

class_weight_dict = get_class_weights(y_train_seq)
print("Class weights:", class_weight_dict)

Class weights: {0: 0.5270464337003351, 1: 9.743362831858407}


In [11]:
# ============================================================
# 9. Model builders
# ============================================================

def make_callbacks(patience=5):
    return [
        EarlyStopping(monitor="val_auc", mode="max", patience=patience, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=max(2, patience // 2), min_lr=1e-5, verbose=1)
    ]


def build_lstm_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(64, return_sequences=False, dropout=0.2, recurrent_dropout=0.0),
        layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name="auc")])
    return model


def build_bilstm_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Bidirectional(layers.LSTM(64, return_sequences=False, dropout=0.2, recurrent_dropout=0.0)),
        layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name="auc")])
    return model


def build_dlstm_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(64, return_sequences=True, dropout=0.2),
        layers.Dropout(0.3),
        layers.LSTM(32, return_sequences=False, dropout=0.2),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name="auc")])
    return model


def transformer_encoder_block(x, num_heads=4, key_dim=32, ff_dim=128, dropout=0.20):
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, dropout=dropout)(x, x)
    x = layers.Add()([x, attn])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    ff = layers.Dense(ff_dim, activation="relu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(x.shape[-1])(ff)
    x = layers.Add()([x, ff])
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return x


def build_transformer_model(input_shape, num_blocks=2, num_heads=4, key_dim=32, ff_dim=128, dropout=0.20, dense_units=64):
    inp = layers.Input(shape=input_shape, name="sequence_input")
    x = layers.Dense(64, activation="relu", name="feature_projection")(inp)
    for _ in range(num_blocks):
        x = transformer_encoder_block(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout=dropout)
    x = layers.GlobalAveragePooling1D(name="temporal_pooling")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation="sigmoid", name="flare_probability")(x)
    model = models.Model(inp, out, name="Transformer_72h_EvolutionAware")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name="auc")])
    return model


def moving_average_decomposition(X, window=5):
    trend = np.zeros_like(X)
    kernel = np.ones(window, dtype=np.float32) / float(window)
    for feature_idx in range(X.shape[2]):
        for sample_idx in range(X.shape[0]):
            trend[sample_idx, :, feature_idx] = np.convolve(X[sample_idx, :, feature_idx], kernel, mode="same")
    residual = X - trend
    return trend, residual

In [12]:
# ============================================================
# 10. Train LSTM
# ============================================================

input_shape = (X_train_scaled.shape[1], X_train_scaled.shape[2])

lstm_model = build_lstm_model(input_shape)
lstm_model.summary()

history_lstm = lstm_model.fit(
    X_train_scaled, y_train_seq,
    validation_data=(X_val_scaled, y_val_seq),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=make_callbacks(5),
    shuffle=False,
    verbose=1
)

val_prob_lstm = lstm_model.predict(X_val_scaled, verbose=0).ravel()
test_prob_lstm = lstm_model.predict(X_test_scaled, verbose=0).ravel()

results_lstm, y_pred_lstm = evaluate_probs_clean("LSTM", y_val_seq, val_prob_lstm, y_test_seq, test_prob_lstm)
display(pd.DataFrame([results_lstm]))

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,945 (105.25 KB)

 Trainable params: 26,945 (105.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - auc: 0.8054 - loss: 0.5626 - val_auc: 0.8452 - val_loss: 0.6663 - learning_rate: 0.0010
Epoch 2/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - auc: 0.8486 - loss: 0.4914 - val_auc: 0.8506 - val_loss: 0.6164 - learning_rate: 0.0010
Epoch 3/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - auc: 0.8582 - loss: 0.4752 - val_auc: 0.8524 - val_loss: 0.5798 - learning_rate: 0.0010
Epoch 4/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - auc: 0.8710 - loss: 0.4552 - val_auc: 0.8483 - val_loss: 0.5730 - learning_rate: 0.0010
Epoch 5/20
51/52 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - auc: 0.8488 - loss: 0.4217
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - auc: 0.8749 - loss: 0.4487 - val_auc: 0.8495 - val_loss: 0.5951 - learning_rate: 0.0010
Epoch 6/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - auc: 0.8848 - loss: 0.4348 - val_auc: 0.8453 - val_loss: 0.5522 - learning_rate: 5.0000e-04
Epoch

,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,LSTM,0.674,0.591951,0.569971,0.327466,0.85949,0.378654,0.793333,0.29602,0.431159,984,283,31,119


In [13]:
# ============================================================
# 11. Train BiLSTM
# ============================================================

bilstm_model = build_bilstm_model(input_shape)
bilstm_model.summary()

history_bilstm = bilstm_model.fit(
    X_train_scaled, y_train_seq,
    validation_data=(X_val_scaled, y_val_seq),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=make_callbacks(5),
    shuffle=False,
    verbose=1
)

val_prob_bilstm = bilstm_model.predict(X_val_scaled, verbose=0).ravel()
test_prob_bilstm = bilstm_model.predict(X_test_scaled, verbose=0).ravel()

results_bilstm, y_pred_bilstm = evaluate_probs_clean("BiLSTM", y_val_seq, val_prob_bilstm, y_test_seq, test_prob_bilstm)
display(pd.DataFrame([results_bilstm]))

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 128)            │        49,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,825 (210.25 KB)

 Trainable params: 53,825 (210.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 18s 80ms/step - auc: 0.8058 - loss: 0.5520 - val_auc: 0.8356 - val_loss: 0.6496 - learning_rate: 0.0010
Epoch 2/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - auc: 0.8519 - loss: 0.4819 - val_auc: 0.8399 - val_loss: 0.6188 - learning_rate: 0.0010
Epoch 3/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - auc: 0.8665 - loss: 0.4637 - val_auc: 0.8447 - val_loss: 0.5940 - learning_rate: 0.0010
Epoch 4/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - auc: 0.8803 - loss: 0.4414 - val_auc: 0.8402 - val_loss: 0.6014 - learning_rate: 0.0010
Epoch 5/20
49/52 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - auc: 0.8509 - loss: 0.4183
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - auc: 0.8810 - loss: 0.4402 - val_auc: 0.8412 - val_loss: 0.6054 - learning_rate: 0.0010
Epoch 6/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - auc: 0.8972 - loss: 0.4129 - val_auc: 0.8383 - val_loss: 0.5392 - learning_rate: 5.0000e-04
Epoc

,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,BiLSTM,0.724,0.580191,0.557948,0.338866,0.85889,0.455232,0.76,0.308108,0.438462,1011,256,36,114


In [14]:
# ============================================================
# 12. Train Transformer
# ============================================================

transformer_model = build_transformer_model(input_shape)
transformer_model.summary()

history_transformer = transformer_model.fit(
    X_train_scaled, y_train_seq,
    validation_data=(X_val_scaled, y_val_seq),
    epochs=30, batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=make_callbacks(6),
    shuffle=False,
    verbose=1
)

val_prob_transformer = transformer_model.predict(X_val_scaled, verbose=0).ravel()
test_prob_transformer = transformer_model.predict(X_test_scaled, verbose=0).ravel()

results_transformer, y_pred_transformer = evaluate_probs_clean("Transformer", y_val_seq, val_prob_transformer, y_test_seq, test_prob_transformer)
display(pd.DataFrame([results_transformer]))

Model: "Transformer_72h_EvolutionAware"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 7, 32)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_projection  │ (None, 7, 64)     │      2,112 │ sequence_input[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 7, 64)     │     33,216 │ feature_projecti… │
│ (MultiHeadAttentio… │                   │            │ feature_projecti… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 7, 64)     │          0 │ feature_projecti… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 7, 64)     │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 7, 128)    │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 7, 128)    │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 7, 64)     │      8,256 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 7, 64)     │          0 │ layer_normalizat… │
│                     │                   │            │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 7, 64)     │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 7, 64)     │     33,216 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 7, 64)     │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 7, 64)     │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 7, 128)    │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 7, 128)    │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 7, 64)     │      8,256 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 7, 64)     │          0 │ layer_normalizat… │
│                     │                   │            │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 7, 64)     │        128 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ temporal_pooling    │ (None, 64)        │          0 │ layer_normalizat

 Total params: 106,433 (415.75 KB)

 Trainable params: 106,433 (415.75 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 14s 92ms/step - auc: 0.7611 - loss: 0.6015 - val_auc: 0.8339 - val_loss: 0.7612 - learning_rate: 0.0010
Epoch 2/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 7s 128ms/step - auc: 0.8281 - loss: 0.5138 - val_auc: 0.8392 - val_loss: 0.7340 - learning_rate: 0.0010
Epoch 3/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 8s 80ms/step - auc: 0.8464 - loss: 0.4917 - val_auc: 0.8397 - val_loss: 0.6337 - learning_rate: 0.0010
Epoch 4/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 7s 139ms/step - auc: 0.8587 - loss: 0.4714 - val_auc: 0.8343 - val_loss: 0.6728 - learning_rate: 0.0010
Epoch 5/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 11s 205ms/step - auc: 0.8698 - loss: 0.4564 - val_auc: 0.8330 - val_loss: 0.6631 - learning_rate: 0.0010
Epoch 6/30
51/52 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - auc: 0.8604 - loss: 0.4015
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
52/52 ━━━━━━━━━━━━━━━━━━━━ 4s 79ms/step - auc: 0.8856 - loss: 0.4267 - val_auc: 0.8272 - val_loss: 0.5711 - learning_rate: 0.0010
Epoc

,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,Transformer,0.714,0.569043,0.536906,0.296938,0.848066,0.379847,0.78,0.275294,0.406957,959,308,33,117


In [15]:
# ============================================================
# 13. DLSTM decomposition and training
# ============================================================

trend_train, res_train = moving_average_decomposition(X_train_scaled, window=MA_WINDOW)
trend_val, res_val = moving_average_decomposition(X_val_scaled, window=MA_WINDOW)
trend_test, res_test = moving_average_decomposition(X_test_scaled, window=MA_WINDOW)

X_train_dlstm = np.concatenate([trend_train, res_train], axis=2)
X_val_dlstm = np.concatenate([trend_val, res_val], axis=2)
X_test_dlstm = np.concatenate([trend_test, res_test], axis=2)

print("DLSTM input shape:", X_train_dlstm.shape)

dlstm_model = build_dlstm_model((X_train_dlstm.shape[1], X_train_dlstm.shape[2]))
dlstm_model.summary()

history_dlstm = dlstm_model.fit(
    X_train_dlstm, y_train_seq,
    validation_data=(X_val_dlstm, y_val_seq),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=make_callbacks(5),
    shuffle=False,
    verbose=1
)

val_prob_dlstm = dlstm_model.predict(X_val_dlstm, verbose=0).ravel()
test_prob_dlstm = dlstm_model.predict(X_test_dlstm, verbose=0).ravel()

results_dlstm, y_pred_dlstm = evaluate_probs_clean("DLSTM", y_val_seq, val_prob_dlstm, y_test_seq, test_prob_dlstm)
display(pd.DataFrame([results_dlstm]))

DLSTM input shape: (6606, 7, 64)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 7, 64)          │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 7, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 45,473 (177.63 KB)

 Trainable params: 45,473 (177.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 8s 35ms/step - auc: 0.7827 - loss: 0.5782 - val_auc: 0.8179 - val_loss: 0.6666 - learning_rate: 0.0010
Epoch 2/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - auc: 0.8288 - loss: 0.5119 - val_auc: 0.8292 - val_loss: 0.6526 - learning_rate: 0.0010
Epoch 3/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 57ms/step - auc: 0.8440 - loss: 0.4885 - val_auc: 0.8351 - val_loss: 0.6190 - learning_rate: 0.0010
Epoch 4/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 4s 66ms/step - auc: 0.8422 - loss: 0.4906 - val_auc: 0.8391 - val_loss: 0.6223 - learning_rate: 0.0010
Epoch 5/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - auc: 0.8573 - loss: 0.4724 - val_auc: 0.8335 - val_loss: 0.5974 - learning_rate: 0.0010
Epoch 6/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - auc: 0.8731 - loss: 0.4517 - val_auc: 0.8397 - val_loss: 0.5849 - learning_rate: 0.0010
Epoch 7/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - auc: 0.8652 - loss: 0.4579 - val_auc: 0.8405 - val_loss: 0.5880 - learning_rate: 0.0010
Epoch 8/20
52

,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,DLSTM,0.675,0.590643,0.574186,0.317556,0.860058,0.399665,0.813333,0.287059,0.424348,964,303,28,122


In [16]:
# ============================================================
# 14. Final clean 72h main table: individual + ensembles
# Validation-selected ensembles only — no test-set model selection
# ============================================================

final_rows = [results_lstm, results_bilstm, results_transformer, results_dlstm]

# ------------------------------------------------------------
# Simple average ensemble: fixed equal-weight ensemble, not tuned by test
# ------------------------------------------------------------
val_prob_simple = (val_prob_lstm + val_prob_bilstm + val_prob_dlstm) / 3.0
test_prob_simple = (test_prob_lstm + test_prob_bilstm + test_prob_dlstm) / 3.0

results_simple, y_pred_simple = evaluate_probs_clean(
    "Simple/Avg Ensemble", y_val_seq, val_prob_simple, y_test_seq, test_prob_simple
)
final_rows.append(results_simple)

# ------------------------------------------------------------
# Weighted ensemble search
# IMPORTANT: choose the weight setting using Val_TSS only.
# Test_TSS is reported only after the validation-selected weight is locked.
# ------------------------------------------------------------
weight_sets = [
    (1/3, 1/3, 1/3),
    (0.2, 0.3, 0.5),
    (0.2, 0.2, 0.6),
    (0.1, 0.3, 0.6),
    (0.1, 0.2, 0.7),
    (0.15, 0.25, 0.60),
    (0.25, 0.25, 0.50),
]

weighted_rows = []
weighted_preds = {}
weighted_probs = {}

for w_lstm, w_bilstm, w_dlstm in weight_sets:
    val_prob_weighted = w_lstm * val_prob_lstm + w_bilstm * val_prob_bilstm + w_dlstm * val_prob_dlstm
    test_prob_weighted = w_lstm * test_prob_lstm + w_bilstm * test_prob_bilstm + w_dlstm * test_prob_dlstm
    model_name = f"Weighted Ensemble ({w_lstm:.2f},{w_bilstm:.2f},{w_dlstm:.2f})"

    result, y_pred_weighted = evaluate_probs_clean(
        model_name, y_val_seq, val_prob_weighted, y_test_seq, test_prob_weighted
    )
    weighted_rows.append(result)
    weighted_preds[model_name] = y_pred_weighted
    weighted_probs[model_name] = (val_prob_weighted, test_prob_weighted)

weighted_df = pd.DataFrame(weighted_rows)
weighted_df_by_val = weighted_df.sort_values("Val_TSS", ascending=False).reset_index(drop=True)
weighted_df_by_test = weighted_df.sort_values("Test_TSS", ascending=False).reset_index(drop=True)

best_weighted_result = weighted_df_by_val.iloc[0].to_dict()
best_weighted_name = best_weighted_result["Model"]
y_pred_best_weighted = weighted_preds[best_weighted_name]
val_prob_best_weighted, test_prob_best_weighted = weighted_probs[best_weighted_name]
final_rows.append(best_weighted_result)

# ------------------------------------------------------------
# Stacking ensemble trained on validation meta-features
# ------------------------------------------------------------
X_meta_val = np.column_stack([val_prob_lstm, val_prob_bilstm, val_prob_dlstm])
X_meta_test = np.column_stack([test_prob_lstm, test_prob_bilstm, test_prob_dlstm])

stack_model = LogisticRegression(class_weight="balanced", max_iter=3000, random_state=SEED)
stack_model.fit(X_meta_val, y_val_seq)

val_prob_stack = stack_model.predict_proba(X_meta_val)[:, 1]
test_prob_stack = stack_model.predict_proba(X_meta_test)[:, 1]

results_stack, y_pred_stack = evaluate_probs_clean(
    "Stacking Ensemble", y_val_seq, val_prob_stack, y_test_seq, test_prob_stack
)
final_rows.append(results_stack)

final_72h_main_df = pd.DataFrame(final_rows).sort_values("Test_TSS", ascending=False).reset_index(drop=True)

display(final_72h_main_df.round(6))
print("\nWeighted ensemble search selected by validation TSS:")
display(weighted_df_by_val.round(6))
print("\nFor audit only: weighted ensemble ranking by test TSS, not used for selection:")
display(weighted_df_by_test.round(6))


,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,Simple/Avg Ensemble,0.707,0.597797,0.593565,0.347474,0.861016,0.430048,0.806667,0.309463,0.447320,997,270,29,121
1,"Weighted Ensemble (0.10,0.20,0.70)",0.695,0.600227,0.582515,0.332986,0.860900,0.423696,0.806667,0.298765,0.436036,983,284,29,121
2,DLSTM,0.675,0.590643,0.574186,0.317556,0.860058,0.399665,0.813333,0.287059,0.424348,964,303,28,122
3,LSTM,0.674,0.591951,0.569971,0.327466,0.859490,0.378654,0.793333,0.296020,0.431159,984,283,31,119
4,Stacking Ensemble,0.633,0.592750,0.564883,0.326195,0.860153,0.389477,0.786667,0.295739,0.429872,986,281,32,118
5,BiLSTM,0.724,0.580191,0.557948,0.338866,0.858890,0.455232,0.760000,0.308108,0.438462,1011,256,36,114
6,Transformer,0.714,0.569043,0.536906,0.296938,0.848066,0.379847,0.780000,0.275294,0.406957,959,308,33,117



Weighted ensemble search selected by validation TSS:


,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,"Weighted Ensemble (0.10,0.20,0.70)",0.695,0.600227,0.582515,0.332986,0.860900,0.423696,0.806667,0.298765,0.436036,983,284,29,121
1,"Weighted Ensemble (0.15,0.25,0.60)",0.688,0.597831,0.577780,0.326983,0.861060,0.425598,0.806667,0.294404,0.431373,977,290,29,121
2,"Weighted Ensemble (0.20,0.20,0.60)",0.686,0.597831,0.579358,0.328971,0.861052,0.417944,0.806667,0.295844,0.432916,979,288,29,121
3,"Weighted Ensemble (0.33,0.33,0.33)",0.707,0.597797,0.593565,0.347474,0.861016,0.430048,0.806667,0.309463,0.447320,997,270,29,121
4,"Weighted Ensemble (0.10,0.30,0.60)",0.690,0.596234,0.577780,0.326983,0.861058,0.431796,0.806667,0.294404,0.431373,977,290,29,121
5,"Weighted Ensemble (0.25,0.25,0.50)",0.701,0.595945,0.588829,0.341180,0.861142,0.424288,0.806667,0.304786,0.442413,991,276,29,121
6,"Weighted Ensemble (0.20,0.30,0.50)",0.707,0.593803,0.588829,0.341180,0.861194,0.428147,0.806667,0.304786,0.442413,991,276,29,121



For audit only: weighted ensemble ranking by test TSS, not used for selection:


,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,"Weighted Ensemble (0.33,0.33,0.33)",0.707,0.597797,0.593565,0.347474,0.861016,0.430048,0.806667,0.309463,0.447320,997,270,29,121
1,"Weighted Ensemble (0.20,0.30,0.50)",0.707,0.593803,0.588829,0.341180,0.861194,0.428147,0.806667,0.304786,0.442413,991,276,29,121
2,"Weighted Ensemble (0.25,0.25,0.50)",0.701,0.595945,0.588829,0.341180,0.861142,0.424288,0.806667,0.304786,0.442413,991,276,29,121
3,"Weighted Ensemble (0.10,0.20,0.70)",0.695,0.600227,0.582515,0.332986,0.860900,0.423696,0.806667,0.298765,0.436036,983,284,29,121
4,"Weighted Ensemble (0.20,0.20,0.60)",0.686,0.597831,0.579358,0.328971,0.861052,0.417944,0.806667,0.295844,0.432916,979,288,29,121
5,"Weighted Ensemble (0.10,0.30,0.60)",0.690,0.596234,0.577780,0.326983,0.861058,0.431796,0.806667,0.294404,0.431373,977,290,29,121
6,"Weighted Ensemble (0.15,0.25,0.60)",0.688,0.597831,0.577780,0.326983,0.861060,0.425598,0.806667,0.294404,0.431373,977,290,29,121


## 15A. Physics-Informed / Physics-Regularised DLSTM Extension

This section adds a PI-DLSTM model. It does **not** claim to solve MHD equations. Instead, it adds an auxiliary physics-proxy loss derived only from past/current SHARP quantities that are physically related to magnetic energy build-up, current/helicity, shear, flux, and PIL activity.

The PI-DLSTM objective is:

\[
\mathcal{L} = \mathcal{L}_{flare} + \lambda_{phys}\mathcal{L}_{physics-proxy}
\]

where \(\mathcal{L}_{flare}\) is binary cross-entropy for M/X-class flare prediction and \(\mathcal{L}_{physics-proxy}\) encourages the latent representation to preserve a SHARP-derived magnetic-complexity proxy.

In [17]:
# ============================================================
# 15A. Build past-only SHARP physics proxy for PI-DLSTM
# ============================================================

from sklearn.preprocessing import StandardScaler

PHYSICS_PROXY_FEATURES = [
    "USFLUX",   # unsigned magnetic flux
    "TOTPOT",   # total photospheric magnetic free-energy density proxy
    "TOTUSJH",  # total unsigned current helicity proxy
    "TOTUSJZ",  # total unsigned vertical current
    "ABSNJZH",  # absolute net current helicity
    "R_VALUE",  # strong-gradient PIL-related flux proxy
    "MEANSHR",  # mean shear angle
    "SHRGT45",  # fraction of pixels with shear > 45 degrees
]

available_proxy_features = [f for f in PHYSICS_PROXY_FEATURES if f in EVO_FEATURES]
proxy_idx = [EVO_FEATURES.index(f) for f in available_proxy_features]

print("Physics proxy features used:", available_proxy_features)
print("Number of physics proxy features:", len(proxy_idx))

assert len(proxy_idx) >= 4, "Too few physics proxy features found. Check EVO_FEATURES and MAG_FEATURES."


def build_physics_proxy_matrix(X_raw_seq, proxy_indices):
    """
    Build a past-only physics proxy matrix from raw, unscaled SHARP sequences.
    Uses only the input sequence, never future flare labels.

    Components:
    1. Last-state magnetic complexity level.
    2. Positive trend from first to last sampled state.
    """
    X_sel = X_raw_seq[:, :, proxy_indices].astype(np.float64)

    # Heavy-tailed SHARP quantities are log-compressed.
    X_log = np.log1p(np.abs(X_sel))

    last_level = X_log[:, -1, :]
    trend = X_log[:, -1, :] - X_log[:, 0, :]
    positive_trend = np.maximum(trend, 0.0)

    proxy_matrix = np.concatenate([last_level, positive_trend], axis=1).astype(np.float32)
    return proxy_matrix


proxy_train_matrix = build_physics_proxy_matrix(X_train_seq, proxy_idx)
proxy_val_matrix = build_physics_proxy_matrix(X_val_seq, proxy_idx)
proxy_test_matrix = build_physics_proxy_matrix(X_test_seq, proxy_idx)

proxy_scaler = StandardScaler()
proxy_train_z = proxy_scaler.fit_transform(proxy_train_matrix)
proxy_val_z = proxy_scaler.transform(proxy_val_matrix)
proxy_test_z = proxy_scaler.transform(proxy_test_matrix)

# Convert the multi-dimensional proxy into a scalar physics-complexity target.
train_score = proxy_train_z.mean(axis=1)
val_score = proxy_val_z.mean(axis=1)
test_score = proxy_test_z.mean(axis=1)

score_mean = train_score.mean()
score_std = train_score.std() + 1e-9


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

physics_train = sigmoid((train_score - score_mean) / score_std).astype(np.float32).reshape(-1, 1)
physics_val = sigmoid((val_score - score_mean) / score_std).astype(np.float32).reshape(-1, 1)
physics_test = sigmoid((test_score - score_mean) / score_std).astype(np.float32).reshape(-1, 1)

print("Physics proxy target ranges:")
print("Train:", float(physics_train.min()), float(physics_train.max()), float(physics_train.mean()))
print("Val  :", float(physics_val.min()), float(physics_val.max()), float(physics_val.mean()))
print("Test :", float(physics_test.min()), float(physics_test.max()), float(physics_test.mean()))

joblib.dump(proxy_scaler, os.path.join(OUT_DIR, "physics_proxy_scaler_72h.joblib"))


Physics proxy features used: ['USFLUX', 'TOTPOT', 'TOTUSJH', 'TOTUSJZ', 'ABSNJZH', 'R_VALUE', 'MEANSHR', 'SHRGT45']
Number of physics proxy features: 8
Physics proxy target ranges:
Train: 0.03669443726539612 0.9956278800964355 0.49295440316200256
Val  : 0.05590929463505745 0.9870648384094238 0.5228301882743835
Test : 0.07336156815290451 0.9877011179924011 0.5351552367210388


['/content/drive/MyDrive/AR_Stratified/Journal_Revision_72h_PI_DLSTM_CLEAN/physics_proxy_scaler_72h.joblib']

In [20]:
# ============================================================
# 15B. Build and train PI-DLSTM — FIXED VERSION
# ============================================================

LAMBDA_PHYS = 0.10


def build_pi_dlstm_model(input_shape, lambda_phys=0.10):
    inp = layers.Input(shape=input_shape, name="pi_dlstm_input")

    x = layers.LSTM(
        64,
        return_sequences=True,
        dropout=0.2,
        name="pi_lstm_trend_residual_1"
    )(inp)

    x = layers.Dropout(0.3)(x)

    x = layers.LSTM(
        32,
        return_sequences=False,
        dropout=0.2,
        name="pi_lstm_trend_residual_2"
    )(x)

    x = layers.Dropout(0.3)(x)

    shared = layers.Dense(
        32,
        activation="relu",
        name="shared_physics_latent"
    )(x)

    shared = layers.Dropout(0.2)(shared)

    flare_out = layers.Dense(
        1,
        activation="sigmoid",
        name="flare_output"
    )(shared)

    physics_hidden = layers.Dense(
        16,
        activation="relu",
        name="physics_proxy_hidden"
    )(shared)

    physics_out = layers.Dense(
        1,
        activation="sigmoid",
        name="physics_proxy_output"
    )(physics_hidden)

    model = models.Model(
        inputs=inp,
        outputs=[flare_out, physics_out],
        name="PI_DLSTM_72h_EvolutionAware"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=[
            "binary_crossentropy",
            "mse"
        ],
        loss_weights=[
            1.0,
            float(lambda_phys)
        ],
        metrics=[
            [tf.keras.metrics.AUC(name="auc")],
            [tf.keras.metrics.MeanSquaredError(name="mse")]
        ]
    )

    return model


pi_dlstm_model = build_pi_dlstm_model(
    input_shape=(X_train_dlstm.shape[1], X_train_dlstm.shape[2]),
    lambda_phys=LAMBDA_PHYS
)

pi_dlstm_model.summary()
print("Model output names:", pi_dlstm_model.output_names)


# ============================================================
# Prepare PI-DLSTM targets
# ============================================================

y_train_flare = np.array(y_train_seq).astype(np.float32).reshape(-1, 1)
y_val_flare = np.array(y_val_seq).astype(np.float32).reshape(-1, 1)

physics_train_target = np.array(physics_train).astype(np.float32).reshape(-1, 1)
physics_val_target = np.array(physics_val).astype(np.float32).reshape(-1, 1)


# ============================================================
# Sample weights
# ============================================================

sample_weight_flare = np.where(
    y_train_seq == 1,
    class_weight_dict[1],
    class_weight_dict[0]
).astype(np.float32)

sample_weight_physics = np.ones(len(y_train_seq), dtype=np.float32)


# ============================================================
# Callbacks
# ============================================================

callbacks_pi = [
    EarlyStopping(
        monitor="val_flare_output_auc",
        mode="max",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_flare_output_auc",
        mode="max",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )
]


# ============================================================
# Train PI-DLSTM
# ============================================================

history_pi_dlstm = pi_dlstm_model.fit(
    X_train_dlstm,
    [
        y_train_flare,
        physics_train_target
    ],
    validation_data=(
        X_val_dlstm,
        [
            y_val_flare,
            physics_val_target
        ]
    ),
    sample_weight=[
        sample_weight_flare,
        sample_weight_physics
    ],
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_pi,
    shuffle=False,
    verbose=1
)

Model: "PI_DLSTM_72h_EvolutionAware"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ pi_dlstm_input      │ (None, 7, 64)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pi_lstm_trend_resi… │ (None, 7, 64)     │     33,024 │ pi_dlstm_input[0… │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 7, 64)     │          0 │ pi_lstm_trend_re… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pi_lstm_trend_resi… │ (None, 32)        │     12,416 │ dropout_16[0][0]  │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 32)        │          0 │ pi_lstm_trend_re… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_physics_lat… │ (None, 32)        │      1,056 │ dropout_17[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 32)        │          0 │ shared_physics_l… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ physics_proxy_hidd… │ (None, 16)        │        528 │ dropout_18[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flare_output        │ (None, 1)         │         33 │ dropout_18[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ physics_proxy_outp… │ (None, 1)         │         17 │ physics_proxy_hi… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 47,074 (183.88 KB)

 Trainable params: 47,074 (183.88 KB)

 Non-trainable params: 0 (0.00 B)

Model output names: ListWrapper(['flare_output', 'physics_proxy_output'])
Epoch 1/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 16s 84ms/step - flare_output_auc: 0.7765 - flare_output_loss: 0.6061 - loss: 0.6045 - physics_proxy_output_loss: 0.0367 - physics_proxy_output_mse: 0.0367 - val_flare_output_auc: 0.8201 - val_flare_output_loss: 0.7089 - val_loss: 0.7001 - val_physics_proxy_output_loss: 0.0277 - val_physics_proxy_output_mse: 0.0258 - learning_rate: 0.0010
Epoch 2/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 4s 72ms/step - flare_output_auc: 0.8232 - flare_output_loss: 0.5243 - loss: 0.5218 - physics_proxy_output_loss: 0.0281 - physics_proxy_output_mse: 0.0280 - val_flare_output_auc: 0.8296 - val_flare_output_loss: 0.6854 - val_loss: 0.6688 - val_physics_proxy_output_loss: 0.0221 - val_physics_proxy_output_mse: 0.0205 - learning_rate: 0.0010
Epoch 3/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 4s 80ms/step - flare_output_auc: 0.8353 - flare_output_loss: 0.5079 - loss: 0.5048 - physics_proxy_output_loss: 0.0241 - physics_prox

In [21]:
# ============================================================
# 15C. Evaluate PI-DLSTM and update final table
# ============================================================

val_pred_outputs = pi_dlstm_model.predict(X_val_dlstm, batch_size=512, verbose=0)
test_pred_outputs = pi_dlstm_model.predict(X_test_dlstm, batch_size=512, verbose=0)

val_prob_pi_dlstm = val_pred_outputs[0].ravel()
val_physics_hat = val_pred_outputs[1].ravel()

test_prob_pi_dlstm = test_pred_outputs[0].ravel()
test_physics_hat = test_pred_outputs[1].ravel()

results_pi_dlstm, y_pred_pi_dlstm = evaluate_probs_clean(
    "PI-DLSTM",
    y_val_seq,
    val_prob_pi_dlstm,
    y_test_seq,
    test_prob_pi_dlstm
)

display(pd.DataFrame([results_pi_dlstm]).round(6))

# Optional fixed ensemble: average of LSTM, BiLSTM, DLSTM, and PI-DLSTM.
val_prob_simple_pi = (val_prob_lstm + val_prob_bilstm + val_prob_dlstm + val_prob_pi_dlstm) / 4.0
test_prob_simple_pi = (test_prob_lstm + test_prob_bilstm + test_prob_dlstm + test_prob_pi_dlstm) / 4.0

results_simple_pi, y_pred_simple_pi = evaluate_probs_clean(
    "Simple/Avg Ensemble + PI-DLSTM",
    y_val_seq,
    val_prob_simple_pi,
    y_test_seq,
    test_prob_simple_pi
)

# Validation-selected weighted ensemble including PI-DLSTM.
# Small predefined grid; selection is by Val_TSS only.
pi_weight_sets = [
    (0.25, 0.25, 0.25, 0.25),
    (0.20, 0.20, 0.30, 0.30),
    (0.15, 0.20, 0.30, 0.35),
    (0.15, 0.15, 0.35, 0.35),
    (0.10, 0.20, 0.30, 0.40),
    (0.10, 0.10, 0.40, 0.40),
]

weighted_pi_rows = []
weighted_pi_preds = {}
weighted_pi_probs = {}

for w_lstm, w_bilstm, w_dlstm, w_pi in pi_weight_sets:
    val_prob_wpi = (
        w_lstm * val_prob_lstm +
        w_bilstm * val_prob_bilstm +
        w_dlstm * val_prob_dlstm +
        w_pi * val_prob_pi_dlstm
    )
    test_prob_wpi = (
        w_lstm * test_prob_lstm +
        w_bilstm * test_prob_bilstm +
        w_dlstm * test_prob_dlstm +
        w_pi * test_prob_pi_dlstm
    )

    model_name = f"Weighted Ensemble + PI ({w_lstm:.2f},{w_bilstm:.2f},{w_dlstm:.2f},{w_pi:.2f})"
    result, y_pred_wpi = evaluate_probs_clean(
        model_name,
        y_val_seq,
        val_prob_wpi,
        y_test_seq,
        test_prob_wpi
    )
    weighted_pi_rows.append(result)
    weighted_pi_preds[model_name] = y_pred_wpi
    weighted_pi_probs[model_name] = (val_prob_wpi, test_prob_wpi)

weighted_pi_df = pd.DataFrame(weighted_pi_rows)
weighted_pi_df_by_val = weighted_pi_df.sort_values("Val_TSS", ascending=False).reset_index(drop=True)
weighted_pi_df_by_test = weighted_pi_df.sort_values("Test_TSS", ascending=False).reset_index(drop=True)

best_weighted_pi_result = weighted_pi_df_by_val.iloc[0].to_dict()
best_weighted_pi_name = best_weighted_pi_result["Model"]
y_pred_best_weighted_pi = weighted_pi_preds[best_weighted_pi_name]
val_prob_best_weighted_pi, test_prob_best_weighted_pi = weighted_pi_probs[best_weighted_pi_name]

# Update final table. Keep the original main rows and add PI extension rows.
pi_extension_rows = [results_pi_dlstm, results_simple_pi, best_weighted_pi_result]

final_72h_main_df = pd.concat(
    [final_72h_main_df, pd.DataFrame(pi_extension_rows)],
    ignore_index=True
).drop_duplicates(subset=["Model"], keep="last")

final_72h_main_df = final_72h_main_df.sort_values("Test_TSS", ascending=False).reset_index(drop=True)

print("Updated 72h main + PI-DLSTM table:")
display(final_72h_main_df.round(6))

print("\nPI-weighted ensemble search selected by validation TSS:")
display(weighted_pi_df_by_val.round(6))

print("\nFor audit only: PI-weighted ensemble ranking by test TSS, not used for selection:")
display(weighted_pi_df_by_test.round(6))


,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,PI-DLSTM,0.761,0.587379,0.540232,0.320666,0.857737,0.425712,0.753333,0.295039,0.424015,997,270,37,113


Updated 72h main + PI-DLSTM table:


,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,Simple/Avg Ensemble,0.707,0.597797,0.593565,0.347474,0.861016,0.430048,0.806667,0.309463,0.447320,997,270,29,121
1,"Weighted Ensemble (0.10,0.20,0.70)",0.695,0.600227,0.582515,0.332986,0.860900,0.423696,0.806667,0.298765,0.436036,983,284,29,121
2,Simple/Avg Ensemble + PI-DLSTM,0.729,0.597508,0.579090,0.344934,0.860852,0.426743,0.786667,0.309711,0.444444,1004,263,32,118
3,DLSTM,0.675,0.590643,0.574186,0.317556,0.860058,0.399665,0.813333,0.287059,0.424348,964,303,28,122
4,LSTM,0.674,0.591951,0.569971,0.327466,0.859490,0.378654,0.793333,0.296020,0.431159,984,283,31,119
5,"Weighted Ensemble + PI (0.10,0.20,0.30,0.40)",0.741,0.599904,0.568913,0.342470,0.860437,0.431736,0.773333,0.309333,0.441905,1008,259,34,116
6,Stacking Ensemble,0.633,0.592750,0.564883,0.326195,0.860153,0.389477,0.786667,0.295739,0.429872,986,281,32,118
7,BiLSTM,0.724,0.580191,0.557948,0.338866,0.858890,0.455232,0.760000,0.308108,0.438462,1011,256,36,114
8,PI-DLSTM,0.761,0.587379,0.540232,0.320666,0.857737,0.425712,0.753333,0.295039,0.424015,997,270,37,113
9,Transformer,0.714,0.569043,0.536906,0.296938,0.848066,0.379847,0.780000,0.275294,0.406957,959,308,33,117



PI-weighted ensemble search selected by validation TSS:


,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,"Weighted Ensemble + PI (0.10,0.20,0.30,0.40)",0.741,0.599904,0.568913,0.342470,0.860437,0.431736,0.773333,0.309333,0.441905,1008,259,34,116
1,"Weighted Ensemble + PI (0.25,0.25,0.25,0.25)",0.729,0.597508,0.579090,0.344934,0.860852,0.426743,0.786667,0.309711,0.444444,1004,263,32,118
2,"Weighted Ensemble + PI (0.15,0.20,0.30,0.35)",0.736,0.597508,0.565756,0.338176,0.860663,0.429414,0.773333,0.306069,0.438563,1004,263,34,116
3,"Weighted Ensemble + PI (0.10,0.10,0.40,0.40)",0.734,0.596454,0.562599,0.333943,0.860274,0.423217,0.773333,0.302872,0.435272,1000,267,34,116
4,"Weighted Ensemble + PI (0.20,0.20,0.30,0.30)",0.729,0.595112,0.576722,0.341730,0.860816,0.427234,0.786667,0.307292,0.441948,1001,266,32,118
5,"Weighted Ensemble + PI (0.15,0.15,0.35,0.35)",0.733,0.595112,0.571634,0.340493,0.860674,0.426182,0.780000,0.307087,0.440678,1003,264,33,117



For audit only: PI-weighted ensemble ranking by test TSS, not used for selection:


,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,"Weighted Ensemble + PI (0.25,0.25,0.25,0.25)",0.729,0.597508,0.579090,0.344934,0.860852,0.426743,0.786667,0.309711,0.444444,1004,263,32,118
1,"Weighted Ensemble + PI (0.20,0.20,0.30,0.30)",0.729,0.595112,0.576722,0.341730,0.860816,0.427234,0.786667,0.307292,0.441948,1001,266,32,118
2,"Weighted Ensemble + PI (0.15,0.15,0.35,0.35)",0.733,0.595112,0.571634,0.340493,0.860674,0.426182,0.780000,0.307087,0.440678,1003,264,33,117
3,"Weighted Ensemble + PI (0.10,0.20,0.30,0.40)",0.741,0.599904,0.568913,0.342470,0.860437,0.431736,0.773333,0.309333,0.441905,1008,259,34,116
4,"Weighted Ensemble + PI (0.15,0.20,0.30,0.35)",0.736,0.597508,0.565756,0.338176,0.860663,0.429414,0.773333,0.306069,0.438563,1004,263,34,116
5,"Weighted Ensemble + PI (0.10,0.10,0.40,0.40)",0.734,0.596454,0.562599,0.333943,0.860274,0.423217,0.773333,0.302872,0.435272,1000,267,34,116


In [22]:
# ============================================================
# 15. Save clean outputs
# ============================================================

final_72h_main_df.to_csv(os.path.join(OUT_DIR, "FINAL_CLEAN_72H_MAIN_RESULTS.csv"), index=False)
weighted_df.to_csv(os.path.join(OUT_DIR, "FINAL_WEIGHTED_ENSEMBLE_SEARCH.csv"), index=False)

np.save(os.path.join(OUT_DIR, "y_val_72h.npy"), y_val_seq)
np.save(os.path.join(OUT_DIR, "y_test_72h.npy"), y_test_seq)

arrays_to_save = {
    "val_prob_lstm": val_prob_lstm,
    "test_prob_lstm": test_prob_lstm,
    "y_pred_lstm": y_pred_lstm,
    "val_prob_bilstm": val_prob_bilstm,
    "test_prob_bilstm": test_prob_bilstm,
    "y_pred_bilstm": y_pred_bilstm,
    "val_prob_transformer": val_prob_transformer,
    "test_prob_transformer": test_prob_transformer,
    "y_pred_transformer": y_pred_transformer,
    "val_prob_dlstm": val_prob_dlstm,
    "test_prob_dlstm": test_prob_dlstm,
    "y_pred_dlstm": y_pred_dlstm,
    "val_prob_simple": val_prob_simple,
    "test_prob_simple": test_prob_simple,
    "y_pred_simple": y_pred_simple,
    "y_pred_best_weighted": y_pred_best_weighted,
    "val_prob_stack": val_prob_stack,
    "test_prob_stack": test_prob_stack,
    "y_pred_stack": y_pred_stack,
}


# Optional PI-DLSTM outputs, if the PI section has been run.
optional_arrays = {
    "val_prob_pi_dlstm": globals().get("val_prob_pi_dlstm", None),
    "test_prob_pi_dlstm": globals().get("test_prob_pi_dlstm", None),
    "y_pred_pi_dlstm": globals().get("y_pred_pi_dlstm", None),
    "val_physics_hat": globals().get("val_physics_hat", None),
    "test_physics_hat": globals().get("test_physics_hat", None),
    "physics_train": globals().get("physics_train", None),
    "physics_val": globals().get("physics_val", None),
    "physics_test": globals().get("physics_test", None),
    "val_prob_simple_pi": globals().get("val_prob_simple_pi", None),
    "test_prob_simple_pi": globals().get("test_prob_simple_pi", None),
    "y_pred_simple_pi": globals().get("y_pred_simple_pi", None),
    "y_pred_best_weighted_pi": globals().get("y_pred_best_weighted_pi", None),
}
for name, arr in optional_arrays.items():
    if arr is not None:
        arrays_to_save[name] = arr

if "weighted_pi_df_by_val" in globals():
    weighted_pi_df_by_val.to_csv(os.path.join(OUT_DIR, "FINAL_WEIGHTED_PI_ENSEMBLE_SEARCH_BY_VAL.csv"), index=False)
if "weighted_df_by_val" in globals():
    weighted_df_by_val.to_csv(os.path.join(OUT_DIR, "FINAL_WEIGHTED_ENSEMBLE_SEARCH_BY_VAL.csv"), index=False)

for name, arr in arrays_to_save.items():
    np.save(os.path.join(OUT_DIR, f"{name}.npy"), arr)

joblib.dump(stack_model, os.path.join(OUT_DIR, "stacking_model_72h.joblib"))
if "pi_dlstm_model" in globals():
    pi_dlstm_model.save(os.path.join(OUT_DIR, "pi_dlstm_72h_model.keras"))

print("Saved all clean outputs to:")
print(OUT_DIR)

Saved all clean outputs to:
/content/drive/MyDrive/AR_Stratified/Journal_Revision_72h_PI_DLSTM_CLEAN


In [23]:
# ============================================================
# 16. Bootstrap confidence interval for best TSS model
# ============================================================

def bootstrap_tss_ci(y_true, y_pred, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.array(y_true).astype(int)
    y_pred = np.array(y_pred).astype(int)
    n = len(y_true)
    scores = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt, yp = y_true[idx], y_pred[idx]
        if len(np.unique(yt)) < 2:
            continue
        cm = confusion_matrix(yt, yp, labels=[0, 1])
        tss, _ = tss_hss_from_cm(cm)
        scores.append(tss)

    scores = np.array(scores)
    return {
        "mean": float(scores.mean()),
        "ci_low": float(np.percentile(scores, 2.5)),
        "ci_high": float(np.percentile(scores, 97.5)),
        "n_boot_valid": int(len(scores))
    }, scores

best_model_name = final_72h_main_df.iloc[0]["Model"]

pred_lookup = {
    "LSTM": y_pred_lstm,
    "BiLSTM": y_pred_bilstm,
    "Transformer": y_pred_transformer,
    "DLSTM": y_pred_dlstm,
    "Simple/Avg Ensemble": y_pred_simple,
    best_weighted_name: y_pred_best_weighted,
    "Stacking Ensemble": y_pred_stack,
}

# Add PI predictions if PI section has been run.
if "y_pred_pi_dlstm" in globals():
    pred_lookup["PI-DLSTM"] = y_pred_pi_dlstm
if "y_pred_simple_pi" in globals():
    pred_lookup["Simple/Avg Ensemble + PI-DLSTM"] = y_pred_simple_pi
if "best_weighted_pi_name" in globals() and "y_pred_best_weighted_pi" in globals():
    pred_lookup[best_weighted_pi_name] = y_pred_best_weighted_pi

best_pred = pred_lookup[best_model_name]
best_ci, best_boot_scores = bootstrap_tss_ci(y_test_seq, best_pred, n_boot=2000, seed=SEED)

print("Best model:", best_model_name)
print("Bootstrap TSS CI:", best_ci)

pd.DataFrame([{
    "Model": best_model_name,
    "Bootstrap_Mean_TSS": best_ci["mean"],
    "CI_Low": best_ci["ci_low"],
    "CI_High": best_ci["ci_high"],
    "N_Boot_Valid": best_ci["n_boot_valid"]
}]).to_csv(os.path.join(OUT_DIR, "BEST_MODEL_BOOTSTRAP_TSS_CI.csv"), index=False)
np.save(os.path.join(OUT_DIR, "best_model_bootstrap_tss_scores.npy"), best_boot_scores)


Best model: Simple/Avg Ensemble
Bootstrap TSS CI: {'mean': 0.5921915514754683, 'ci_low': 0.5258708919731291, 'ci_high': 0.6561059751249115, 'n_boot_valid': 2000}


In [24]:
# ============================================================
# 17. McNemar comparisons: best model vs baselines
# ============================================================

from math import erf, sqrt

def mcnemar_test(y_true, pred_a, pred_b):
    y_true = np.array(y_true).astype(int)
    pred_a = np.array(pred_a).astype(int)
    pred_b = np.array(pred_b).astype(int)

    a_correct = pred_a == y_true
    b_correct = pred_b == y_true

    b = int(np.sum((a_correct == 1) & (b_correct == 0)))
    c = int(np.sum((a_correct == 0) & (b_correct == 1)))
    n = b + c

    if n == 0:
        chi2, p_value = 0.0, 1.0
    else:
        chi2 = ((abs(b - c) - 1) ** 2) / (b + c + 1e-9)
        z = sqrt(chi2)
        p_value = 2 * (1 - 0.5 * (1 + erf(z / sqrt(2))))

    return {"b": b, "c": c, "n": n, "chi2": float(chi2), "p_value": float(p_value)}

mcnemar_baselines = [
    ("Best vs LSTM", y_pred_lstm),
    ("Best vs BiLSTM", y_pred_bilstm),
    ("Best vs Transformer", y_pred_transformer),
    ("Best vs DLSTM", y_pred_dlstm),
    ("Best vs Simple/Avg Ensemble", y_pred_simple),
    ("Best vs Stacking Ensemble", y_pred_stack),
]

if "y_pred_pi_dlstm" in globals():
    mcnemar_baselines.append(("Best vs PI-DLSTM", y_pred_pi_dlstm))
if "y_pred_simple_pi" in globals():
    mcnemar_baselines.append(("Best vs Simple/Avg Ensemble + PI-DLSTM", y_pred_simple_pi))

mcnemar_rows = []

for comparison_name, baseline_pred in mcnemar_baselines:
    if comparison_name.replace("Best vs ", "") == best_model_name:
        continue
    out = mcnemar_test(y_test_seq, best_pred, baseline_pred)
    out["Comparison"] = comparison_name
    out["Best_Model"] = best_model_name
    mcnemar_rows.append(out)

mcnemar_df = pd.DataFrame(mcnemar_rows)[["Comparison", "Best_Model", "b", "c", "n", "chi2", "p_value"]]
display(mcnemar_df)

mcnemar_df.to_csv(os.path.join(OUT_DIR, "MCNEMAR_BEST_MODEL_COMPARISONS.csv"), index=False)


,Comparison,Best_Model,b,c,n,chi2,p_value
0,Best vs LSTM,Simple/Avg Ensemble,22,7,29,6.758621,9.329585e-03
1,Best vs BiLSTM,Simple/Avg Ensemble,13,20,33,1.090909,2.962699e-01
2,Best vs Transformer,Simple/Avg Ensemble,58,16,74,22.716216,1.877765e-06
3,Best vs DLSTM,Simple/Avg Ensemble,34,2,36,26.694444,2.383057e-07
4,Best vs Stacking Ensemble,Simple/Avg Ensemble,17,3,20,8.450000,3.650434e-03
5,Best vs PI-DLSTM,Simple/Avg Ensemble,24,16,40,1.225000,2.683816e-01
6,Best vs Simple/Avg Ensemble + PI-DLSTM,Simple/Avg Ensemble,3,7,10,0.900000,3.427817e-01


In [25]:
# ============================================================
# 18. Manuscript-ready rounded table
# ============================================================

manuscript_cols = [
    "Model", "Threshold", "Val_TSS", "Test_TSS", "HSS",
    "ROC_AUC", "PR_AUC", "Recall", "Precision", "F1",
    "TN", "FP", "FN", "TP"
]

manuscript_table = final_72h_main_df[manuscript_cols].copy()
num_cols = manuscript_table.select_dtypes(include=[np.number]).columns
manuscript_table[num_cols] = manuscript_table[num_cols].round(3)

display(manuscript_table)

manuscript_table.to_csv(os.path.join(OUT_DIR, "MANUSCRIPT_READY_72H_MAIN_TABLE.csv"), index=False)
print("Done. Use MANUSCRIPT_READY_72H_MAIN_TABLE.csv for Table 6.3.")

,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,TN,FP,FN,TP
0,Simple/Avg Ensemble,0.707,0.598,0.594,0.347,0.861,0.430,0.807,0.309,0.447,997,270,29,121
1,"Weighted Ensemble (0.10,0.20,0.70)",0.695,0.600,0.583,0.333,0.861,0.424,0.807,0.299,0.436,983,284,29,121
2,Simple/Avg Ensemble + PI-DLSTM,0.729,0.598,0.579,0.345,0.861,0.427,0.787,0.310,0.444,1004,263,32,118
3,DLSTM,0.675,0.591,0.574,0.318,0.860,0.400,0.813,0.287,0.424,964,303,28,122
4,LSTM,0.674,0.592,0.570,0.327,0.859,0.379,0.793,0.296,0.431,984,283,31,119
5,"Weighted Ensemble + PI (0.10,0.20,0.30,0.40)",0.741,0.600,0.569,0.342,0.860,0.432,0.773,0.309,0.442,1008,259,34,116
6,Stacking Ensemble,0.633,0.593,0.565,0.326,0.860,0.389,0.787,0.296,0.430,986,281,32,118
7,BiLSTM,0.724,0.580,0.558,0.339,0.859,0.455,0.760,0.308,0.438,1011,256,36,114
8,PI-DLSTM,0.761,0.587,0.540,0.321,0.858,0.426,0.753,0.295,0.424,997,270,37,113
9,Transformer,0.714,0.569,0.537,0.297,0.848,0.380,0.780,0.275,0.407,959,308,33,117


Done. Use MANUSCRIPT_READY_72H_MAIN_TABLE.csv for Table 6.3.
